In [3]:
# Parsing the text labels and verifying the text path

import os
import pandas as pd
from pathlib import Path

# Kaggle input path
BASE_INPUT = Path("/kaggle/input/datasets/nibinv23/iam-handwriting-word-database/iam_words")
WORDS_TXT = BASE_INPUT / "words.txt"
WORDS_IMG_DIR = BASE_INPUT / "words"   

# Below code is used store the id and text in a python list as each tuple has text id and the word.
records = []
with open(WORDS_TXT, "r") as f:
    for line in f:
        line = line.strip() # used to clean a string removes spaces and newline
        if not line or line.startswith("#"):
            continue
        parts = line.split() # breaks the line into list of words using whitespace stores in a list named parts.
        if len(parts) < 9:               # The size of parts array is less than 9 then skip the line.
            continue
        word_id = parts[0]               # e.g., a01-000u-00-00
        seg_status = parts[1]            # ok or err
        transcription = " ".join(parts[8:])  # transcription may contain spaces

        # keep only well‑segmented words
        if seg_status != "ok":
            continue

        records.append((word_id, transcription))

# 2. coonverted into  a DataFrame
df = pd.DataFrame(records, columns=["word_id", "text"])

# 3. Build image paths and verify existence
def build_img_path(word_id):
    # The word_id is like 'a01-000u-00-00'
    # The actual file is under words/a01/a01-000u/a01-000u-00-00.png
    parts = word_id.split("-")
    folder1 = parts[0]                     # "a01"
    folder2 = f"{parts[0]}-{parts[1]}"     # "a01-000u"
    filename = f"{word_id}.png"
    return WORDS_IMG_DIR / folder1 / folder2 / filename

df["img_path"] = df["word_id"].apply(build_img_path)
df["exists"] = df["img_path"].apply(lambda p: p.is_file())

# 4. Filter to only existing images
df_valid = df[df["exists"]].copy()

print(f"Total entries in words.txt (ok only): {len(df)}")
print(f"Images found on disk: {len(df_valid)}")
print(f"Missing images: {len(df) - len(df_valid)}")

# Show a few samples
df.head()

Total entries in words.txt (ok only): 38305
Images found on disk: 38305
Missing images: 0


,word_id,text,img_path,exists
0,a01-000u-00-00,A,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
1,a01-000u-00-01,MOVE,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
2,a01-000u-00-02,to,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
3,a01-000u-00-03,stop,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
4,a01-000u-00-04,Mr.,/kaggle/input/datasets/nibinv23/iam-handwritin...,True


In [3]:
# Preprocessing 

from collections import Counter

# Collect all characters from your labels
all_text = ''.join(df_valid['text'].values)
char_counts = Counter(all_text)
chars = sorted(char_counts.keys())

# Add a blank token for CTC (index 0)
chars = ['<blank>'] + chars
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
num_classes = len(chars)

print(f"Number of unique characters: {num_classes}")
print(f"Character set: {chars}")

In [4]:
# TRAINING LOOP

import json
import csv
import time
import h5py
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

# --- CONFIGURATION ---
WORKING_DIR  = Path("/kaggle/working")
H5_PATH      = WORKING_DIR / "iam_processed.h5"
VOCAB_PATH   = WORKING_DIR / "vocab.json"
CKPT_DIR     = WORKING_DIR / "checkpoints"

BATCH_SIZE   = 64
NUM_EPOCHS   = 50
LR           = 1e-3
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- DATASET & DATALOADER ---
class IAMDataset(Dataset):
    def __init__(self, h5_path, split, augment=False):
        self.h5_path = str(h5_path)
        self.split = split
        
        # HandwritingAugmentor is already in memory from your previous Kaggle cell
        self.augmentor = HandwritingAugmentor(p=0.4) if augment else None
        
        self.h5_file = None # Will be opened per-worker to prevent multiprocessing crashes
        
        # Get total length once
        with h5py.File(self.h5_path, "r") as f:
            self.length = f[self.split]["images"].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Open the file only once per worker, preventing massive I/O bottlenecks
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, "r")
            
        img = self.h5_file[self.split]["images"][idx]
        label = self.h5_file[self.split]["labels"][idx]
        text = self.h5_file[self.split]["texts"][idx]

        if self.augmentor:
            img = self.augmentor(img)

        return (
            torch.from_numpy(img).unsqueeze(0), 
            torch.from_numpy(label.astype(np.int64)), 
            text
        )

def collate_fn(batch):
    images, labels, texts = zip(*batch)
    max_w = max(img.shape[2] for img in images)

    # Pad images with white space (1.0) to match the widest image in the batch
    padded_imgs = [torch.nn.functional.pad(img, (0, max_w - img.shape[2]), value=1.0) for img in images]
    
    images_t = torch.stack(padded_imgs)
    targets = torch.cat(list(labels)).long()
    
    target_lens = torch.tensor([len(l) for l in labels], dtype=torch.long)
    input_lens = torch.full((len(images),), max_w // 4, dtype=torch.long) # CNN reduces width by 4

    return images_t, targets, input_lens, target_lens, list(texts)

# --- MODEL ---
class CRNN(nn.Module):
    def __init__(self, vocab_size, rnn_hidden=256):
        super().__init__()
        
        def _block(in_c, out_c, pool=None, bn=False):
            layers = [nn.Conv2d(in_c, out_c, 3, padding=1)]
            if bn: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.ReLU(inplace=True))
            if pool: layers.append(nn.MaxPool2d(*pool))
            return layers

        self.cnn = nn.Sequential(
            *_block(1,   64,  pool=((2, 2),)),
            *_block(64,  128, pool=((2, 2),)),
            *_block(128, 256, bn=True),
            *_block(256, 256, pool=((2, 1),)),
            *_block(256, 512, bn=True),
            *_block(512, 512, pool=((2, 1),)),
            *_block(512, 512, bn=True, pool=((2, 1),)),
        )

        self.rnn = nn.LSTM(512, rnn_hidden, num_layers=2, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        feat = self.cnn(x).squeeze(2).permute(2, 0, 1) # (T, B, 512)
        out, _ = self.rnn(feat)
        return self.fc(out)

# --- METRICS & DECODING ---
def greedy_decode(logits, idx2char, blank=0):
    preds = logits.argmax(2).permute(1, 0)
    decoded = []
    for row in preds.tolist():
        chars, prev = [], blank
        for p in row:
            if p != blank and p != prev:
                chars.append(idx2char.get(p, "?"))
            prev = p
        decoded.append("".join(chars))
    return decoded

def calculate_cer(pred, gt):
    if not gt: return 0.0 if not pred else 1.0
    
    # Standard Levenshtein Distance
    m, n = len(pred), len(gt)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, n + 1):
            dp[j] = prev[j-1] if pred[i-1] == gt[j-1] else 1 + min(prev[j-1], prev[j], dp[j-1])
    return dp[n] / n

# --- TRAINING LOOP ---
def run_epoch(model, loader, criterion, optimizer, idx2char, blank, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, total_cer, batches = 0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, targets, in_lens, tgt_lens, texts in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            in_lens, tgt_lens = in_lens.to(DEVICE), tgt_lens.to(DEVICE)

            logits = model(imgs)
            loss = criterion(logits.log_softmax(2), targets, in_lens, tgt_lens)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

            preds = greedy_decode(logits.detach().cpu(), idx2char, blank)
            total_loss += loss.item()
            total_cer += sum(calculate_cer(p, g) for p, g in zip(preds, texts)) / len(preds)
            batches += 1

    return total_loss / batches, total_cer / batches

def main():
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Using Device: {DEVICE}")

    # Load Vocab
    with open(VOCAB_PATH) as f: vocab = json.load(f)
    idx2char = {int(k): v for k, v in vocab["idx2char"].items()}
    vocab_size, blank = vocab["vocab_size"], vocab["blank_index"]

    # Setup Data
    train_loader = DataLoader(IAMDataset(H5_PATH, "train", augment=True), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader = DataLoader(IAMDataset(H5_PATH, "val"), batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=2)

    # Setup Model & Optimizer
    model = CRNN(vocab_size).to(DEVICE)
    criterion = nn.CTCLoss(blank=blank, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    # Log setup
    log_path = WORKING_DIR / "training_log.csv"
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "val_loss", "train_cer", "val_cer", "lr"])

    best_cer = float("inf")

    # Training Execution
    print("\nStarting Training...")
    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        
        tr_loss, tr_cer = run_epoch(model, train_loader, criterion, optimizer, idx2char, blank, is_train=True)
        va_loss, va_cer = run_epoch(model, val_loader, criterion, optimizer, idx2char, blank, is_train=False)
        
        scheduler.step(va_cer)
        
        print(f"Epoch {epoch:03d} | Train Loss: {tr_loss:.4f} CER: {tr_cer:.4f} | Val Loss: {va_loss:.4f} CER: {va_cer:.4f} | Time: {time.time()-t0:.0f}s")

        # Checkpointing
        checkpoint = {"epoch": epoch, "model": model.state_dict(), "opt": optimizer.state_dict(), "val_cer": va_cer}
        torch.save(checkpoint, CKPT_DIR / "last_model.pt")
        
        if va_cer < best_cer:
            best_cer = va_cer
            torch.save(checkpoint, CKPT_DIR / "best_model.pt")
            print(f" -> New Best Model Saved! (CER: {best_cer:.4f})")

        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch, tr_loss, va_loss, tr_cer, va_cer, optimizer.param_groups[0]['lr']])

if __name__ == "__main__":
    main()

Using Device: cuda

Starting Training...
Epoch 001 | Train Loss: 3.2151 CER: 1.0131 | Val Loss: 2.8099 CER: 1.0236 | Time: 63s
 -> New Best Model Saved! (CER: 1.0236)
Epoch 002 | Train Loss: 2.1617 CER: 1.0081 | Val Loss: 1.9872 CER: 1.0192 | Time: 62s
 -> New Best Model Saved! (CER: 1.0192)
Epoch 003 | Train Loss: 1.7568 CER: 1.0078 | Val Loss: 1.5062 CER: 1.0035 | Time: 61s
 -> New Best Model Saved! (CER: 1.0035)
Epoch 004 | Train Loss: 1.3833 CER: 1.0089 | Val Loss: 1.1112 CER: 1.0165 | Time: 61s
Epoch 005 | Train Loss: 1.0729 CER: 1.0135 | Val Loss: 0.8872 CER: 1.0070 | Time: 61s
Epoch 006 | Train Loss: 0.8822 CER: 1.0131 | Val Loss: 0.7854 CER: 1.0181 | Time: 61s
Epoch 007 | Train Loss: 0.7693 CER: 1.0115 | Val Loss: 0.7164 CER: 1.0169 | Time: 62s
Epoch 008 | Train Loss: 0.6897 CER: 1.0103 | Val Loss: 0.6666 CER: 1.0081 | Time: 61s
Epoch 009 | Train Loss: 0.6334 CER: 1.0102 | Val Loss: 0.5705 CER: 1.0098 | Time: 61s
Epoch 010 | Train Loss: 0.5043 CER: 1.0076 | Val Loss: 0.5148 CER

In [5]:
# TESTING LOOP 

import json
import csv
import math
import h5py
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- CONFIGURATION ---
WORKING_DIR  = Path("/kaggle/working")
H5_PATH      = WORKING_DIR / "iam_processed.h5"
VOCAB_PATH   = WORKING_DIR / "vocab.json"
CKPT_PATH    = WORKING_DIR / "checkpoints" / "best_model.pt"
RESULTS_CSV  = WORKING_DIR / "test_results.csv"
METRICS_JSON = WORKING_DIR / "test_metrics.json"

BATCH_SIZE   = 64
BEAM_WIDTH   = 10
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- OPTIMIZED DATASET ---
class IAMDataset(Dataset):
    def __init__(self, h5_path, split):
        self.h5_path, self.split = str(h5_path), split
        self.h5_file = None
        with h5py.File(self.h5_path, "r") as f:
            self.length = f[split]["images"].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, "r")
        img = self.h5_file[self.split]["images"][idx]
        label = self.h5_file[self.split]["labels"][idx]
        text = self.h5_file[self.split]["texts"][idx]
        # --- FIX 1: decode bytes to str ---
        if isinstance(text, bytes):
            text = text.decode("utf-8")
        return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(label.astype(np.int64)), text


def collate_fn(batch):
    images, labels, texts = zip(*batch)
    max_w = max(img.shape[2] for img in images)
    padded = [torch.nn.functional.pad(img, (0, max_w - img.shape[2]), value=1.0) for img in images]

    images_t = torch.stack(padded)
    targets = torch.cat(list(labels)).long()
    target_lens = torch.tensor([len(l) for l in labels], dtype=torch.long)
    input_lens = torch.full((len(images),), max_w // 4, dtype=torch.long)
    return images_t, targets, input_lens, target_lens, list(texts)


# --- MODEL (unchanged) ---
class CRNN(nn.Module):
    def __init__(self, vocab_size, rnn_hidden=256):
        super().__init__()
        def _block(ic, oc, pool=None, bn=False):
            layers = [nn.Conv2d(ic, oc, 3, padding=1)]
            if bn: layers.append(nn.BatchNorm2d(oc))
            layers.append(nn.ReLU(inplace=True))
            if pool: layers.append(nn.MaxPool2d(*pool))
            return layers

        self.cnn = nn.Sequential(
            *_block(1, 64, pool=((2,2),)), *_block(64, 128, pool=((2,2),)),
            *_block(128, 256, bn=True), *_block(256, 256, pool=((2,1),)),
            *_block(256, 512, bn=True), *_block(512, 512, pool=((2,1),)),
            *_block(512, 512, bn=True, pool=((2,1),))
        )
        self.rnn = nn.LSTM(512, rnn_hidden, 2, bidirectional=True, dropout=0.3)
        self.fc  = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        f = self.cnn(x).squeeze(2).permute(2, 0, 1)
        out, _ = self.rnn(f)
        return self.fc(out)


# --- DECODERS & METRICS ---
def greedy_decode(logits, idx2char, blank=0):
    """
    Standard CTC greedy decoding:
    - take argmax per time step
    - collapse repeated characters (skip blanks)
    """
    _, max_indices = logits.argmax(2).permute(1, 0)  # (B, T)
    decoded = []
    for row in max_indices.tolist():
        chars = []
        prev = blank
        for p in row:
            if p != blank and p != prev:
                chars.append(idx2char.get(p, "?"))
            prev = p
        decoded.append("".join(chars))
    return decoded


def beam_search_decode(logits, idx2char, blank=0, beam_width=10):
    # (unchanged, it's fine)
    log_probs = logits.log_softmax(2).cpu().numpy()
    T, B, V = log_probs.shape
    results = []

    def log_add(a, b):
        if a == float("-inf"): return b
        if b == float("-inf"): return a
        m = max(a, b)
        return m + math.log(1 + math.exp(min(a, b) - m))

    for b in range(B):
        lp = log_probs[:, b, :]
        beams = {(): (0.0, float("-inf"))}  # prefix: (p_blank, p_nonblank)

        for t in range(T):
            new_beams = {}
            for prefix, (p_b, p_nb) in beams.items():
                p_total = log_add(p_b, p_nb)

                for c in range(V):
                    lp_c = lp[t, c]
                    if c == blank:
                        entry = new_beams.get(prefix, (float("-inf"), float("-inf")))
                        new_beams[prefix] = (log_add(entry[0], p_total + lp_c), entry[1])
                    else:
                        new_prefix = prefix + (c,)
                        entry = new_beams.get(new_prefix, (float("-inf"), float("-inf")))

                        if prefix and prefix[-1] == c:
                            new_beams[new_prefix] = (entry[0], log_add(entry[1], p_b + lp_c))
                        else:
                            new_beams[new_prefix] = (entry[0], log_add(entry[1], p_total + lp_c))

            beams = dict(sorted(new_beams.items(), key=lambda x: log_add(x[1][0], x[1][1]), reverse=True)[:beam_width])

        best = max(beams, key=lambda p: log_add(beams[p][0], beams[p][1]))
        results.append("".join(idx2char.get(c, "?") for c in best if c != blank))
    return results


def calculate_edit_distance(a, b):
    dp = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, len(b) + 1):
            dp[j] = prev[j-1] if a[i-1] == b[j-1] else 1 + min(prev[j-1], prev[j], dp[j-1])
    return dp[-1]

def cer(pred, gt): return calculate_edit_distance(pred, gt) / len(gt) if gt else (1.0 if pred else 0.0)
def wer(pred, gt): return calculate_edit_distance(pred.split(), gt.split()) / len(gt.split()) if gt else (1.0 if pred else 0.0)


# --- MAIN EVALUATION ---
def main():
    print(f"Device: {DEVICE}")

    with open(VOCAB_PATH) as f: vocab = json.load(f)
    idx2char = {int(k): v for k, v in vocab["idx2char"].items()}
    vocab_size, blank = vocab["vocab_size"], vocab["blank_index"]

    model = CRNN(vocab_size).to(DEVICE)
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded best checkpoint (Epoch {ckpt['epoch']}, Val CER: {ckpt['val_cer']:.4f})")

    loader = DataLoader(IAMDataset(H5_PATH, "test"), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

    all_gts, all_greedy, all_beam = [], [], []

    print("\nRunning Inference...")
    with torch.no_grad():
        for imgs, _, _, _, texts in tqdm(loader):
            logits = model(imgs.to(DEVICE))
            all_greedy.extend(greedy_decode(logits, idx2char, blank))
            all_beam.extend(beam_search_decode(logits, idx2char, blank, BEAM_WIDTH))
            all_gts.extend(texts)

    # Metrics
    g_cers = [cer(p, g) for p, g in zip(all_greedy, all_gts)]
    b_cers = [cer(p, g) for p, g in zip(all_beam, all_gts)]
    g_wers = [wer(p, g) for p, g in zip(all_greedy, all_gts)]
    b_wers = [wer(p, g) for p, g in zip(all_beam, all_gts)]

    mean_g_cer, mean_b_cer = np.mean(g_cers), np.mean(b_cers)
    mean_g_wer, mean_b_wer = np.mean(g_wers), np.mean(b_wers)

    # Save CSV
    with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Ground Truth", "Greedy Pred", "Beam Pred", "Greedy CER", "Beam CER"])
        for gt, gp, bp, gc, bc in zip(all_gts, all_greedy, all_beam, g_cers, b_cers):
            writer.writerow([gt, gp, bp, f"{gc:.4f}", f"{bc:.4f}"])

    # Save JSON
    with open(METRICS_JSON, "w") as f:
        json.dump({"Greedy CER": mean_g_cer, "Beam CER": mean_b_cer, "Greedy WER": mean_g_wer, "Beam WER": mean_b_wer}, f, indent=2)

    # Print Summary
    print("\n" + "="*50)
    print(f"{'Metric':<15} | {'Greedy':<15} | {'Beam (k='+str(BEAM_WIDTH)+')':<15}")
    print("-" * 50)
    print(f"{'Mean CER':<15} | {mean_g_cer:<15.4f} | {mean_b_cer:<15.4f}")
    print(f"{'Mean WER':<15} | {mean_g_wer:<15.4f} | {mean_b_wer:<15.4f}")
    print("=" * 50)

    # Print 5 Random Predictions
    print("\nRandom Samples:")
    for i in np.random.choice(len(all_gts), 5, replace=False):
        print(f"GT: {all_gts[i]} | Greedy: {all_greedy[i]} | Beam: {all_beam[i]}")

if __name__ == "__main__":
    main()

Device: cuda


KeyError: 'model_state_dict'

In [ ]:
# INFERENCE SCRIPT (USED IF THERE IS ANY NEW FODER OF IMAGES OR NOT)

import json
import csv
import math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

# --- CONFIGURATION ---
WORKING_DIR = Path("/kaggle/working")
VOCAB_PATH  = WORKING_DIR / "vocab.json"
CKPT_PATH   = WORKING_DIR / "checkpoints" / "best_model.pt"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- MODEL (Included for fresh kernel execution) ---
class CRNN(nn.Module):
    def __init__(self, vocab_size, rnn_hidden=256):
        super().__init__()
        def _b(ic, oc, pool=None, bn=False):
            l = [nn.Conv2d(ic, oc, 3, padding=1)]
            if bn: l.append(nn.BatchNorm2d(oc))
            l.append(nn.ReLU(inplace=True))
            if pool: l.append(nn.MaxPool2d(*pool))
            return l
            
        self.cnn = nn.Sequential(
            *_b(1, 64, pool=((2,2),)), *_b(64, 128, pool=((2,2),)),
            *_b(128, 256, bn=True), *_b(256, 256, pool=((2,1),)),
            *_b(256, 512, bn=True), *_b(512, 512, pool=((2,1),)),
            *_b(512, 512, bn=True, pool=((2,1),))
        )
        self.rnn = nn.LSTM(512, rnn_hidden, 2, bidirectional=True, dropout=0.3)
        self.fc  = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        return self.fc(self.rnn(self.cnn(x).squeeze(2).permute(2, 0, 1))[0])

# --- DECODERS ---
def greedy_decode(logits, idx2char, blank=0):
    preds = logits.argmax(2).squeeze(1).tolist()
    return "".join(idx2char.get(p, "?") for i, p in enumerate(preds) if p != blank and (i == 0 or p != preds[i-1]))

def beam_decode(logits, idx2char, blank=0, beam_width=10):
    lp = logits.squeeze(1).log_softmax(1).cpu().numpy()
    beams = {(): (0.0, float("-inf"))}

    def log_add(a, b):
        if a == float("-inf"): return b
        if b == float("-inf"): return a
        m = max(a, b)
        return m + math.log(1.0 + math.exp(min(a, b) - m))

    for t in range(lp.shape[0]):
        new_beams = {}
        for prefix, (p_b, p_nb) in beams.items():
            p_tot = log_add(p_b, p_nb)
            for c in range(lp.shape[1]):
                lpc = lp[t, c]
                if c == blank:
                    e = new_beams.get(prefix, (float("-inf"), float("-inf")))
                    new_beams[prefix] = (log_add(e[0], p_tot + lpc), e[1])
                else:
                    np_ = prefix + (c,)
                    e = new_beams.get(np_, (float("-inf"), float("-inf")))
                    if prefix and prefix[-1] == c:
                        new_beams[np_] = (e[0], log_add(e[1], p_b + lpc))
                    else:
                        new_beams[np_] = (e[0], log_add(e[1], p_tot + lpc))

        beams = dict(sorted(new_beams.items(), key=lambda x: log_add(x[1][0], x[1][1]), reverse=True)[:beam_width])

    best = max(beams, key=lambda p: log_add(beams[p][0], beams[p][1]))
    return "".join(idx2char.get(c, "?") for c in best if c != blank)

# --- IMAGE PREPARATION ---
def load_image(path: str, height=32, width=128):
    img = Image.open(path).convert("L")
    ow, oh = img.size
    nw = max(1, int(ow * (height / oh)))
    img = img.resize((nw, height), Image.LANCZOS)
    
    # Pad to 128 width or crop if longer
    canvas = Image.new("L", (width, height), 255)
    canvas.paste(img.crop((0, 0, min(nw, width), height)), (0, 0))
    
    arr = np.array(canvas, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(DEVICE)

# --- INFERENCE ENGINE ---
def run_inference(image_path=None, folder_path=None, beam_width=10, output_csv="predictions.csv"):
    """
    Pass an image_path (string) to test a single image, 
    or a folder_path (string) to test a whole directory.
    """
    # 1. Load Vocab & Model
    with open(VOCAB_PATH) as f: vocab = json.load(f)
    idx2char = {int(k): v for k, v in vocab["idx2char"].items()}
    
    model = CRNN(vocab["vocab_size"]).to(DEVICE)
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE)["model_state_dict"])
    model.eval()

    # 2. Single Image Mode
    if image_path:
        print(f"\nProcessing Single Image: {Path(image_path).name}")
        with torch.no_grad():
            logits = model(load_image(image_path))
            
        print(f"Greedy : {greedy_decode(logits.cpu(), idx2char, vocab['blank_index'])}")
        print(f"Beam   : {beam_decode(logits.cpu(), idx2char, vocab['blank_index'], beam_width)}")
        return

    # 3. Folder Mode
    if folder_path:
        folder = Path(folder_path)
        paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
        print(f"\nProcessing {len(paths)} images from {folder.name}...")
        
        results = []
        for path in paths:
            try:
                with torch.no_grad():
                    logits = model(load_image(str(path)))
                g = greedy_decode(logits.cpu(), idx2char, vocab["blank_index"])
                b = beam_decode(logits.cpu(), idx2char, vocab["blank_index"], beam_width)
            except Exception as e:
                g = b = f"ERROR: {e}"
                
            results.append({"file": path.name, "greedy": g, "beam": b})
            print(f"{path.name:<20} | greedy: {g:<15} | beam: {b}")

        # Save to CSV
        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["file", "greedy", "beam"])
            writer.writeheader()
            writer.writerows(results)
        print(f"\nSaved {len(results)} predictions -> {output_csv}")

In [ ]:
# VISUALZATION

"""

        5_visualise.py  —  Training Curves + Error Analysis                 

 Reads:                                                                      
   /kaggle/working/training_log.csv    — loss + CER per epoch                
   /kaggle/working/test_results.csv    — per-sample predictions              
                                                                             
 Produces  /kaggle/working/figures/                                          
 01_loss_curves.png     train vs val loss                                 
 02_cer_curves.png      train vs val CER                                   
 03_lr_schedule.png     learning-rate over epochs                          
 04_cer_histogram.png   test-set CER distribution (greedy vs beam)         
05_error_analysis.png  top-20 most confused character pairs               

"""

import json
import csv
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                   # no display needed on Kaggle
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── CONFIG ────────────────────────────────────────────────────────────────────
WORKING_DIR  = Path("/kaggle/working")
LOG_CSV      = WORKING_DIR / "training_log.csv"
RESULTS_CSV  = WORKING_DIR / "test_results.csv"
FIG_DIR      = WORKING_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

STYLE_COLOR  = "#2563EB"      # blue
STYLE_ORANGE = "#EA580C"      # orange
# ─────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 120,
})


def load_log():
    df = pd.read_csv(LOG_CSV)
    # normalise column names (strip whitespace)
    df.columns = [c.strip() for c in df.columns]
    return df


def plot_loss(df):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["epoch"], df["train_loss"], label="Train loss", color=STYLE_COLOR)
    ax.plot(df["epoch"], df["val_loss"],   label="Val loss",   color=STYLE_ORANGE)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("CTC Loss")
    ax.set_title("Training vs Validation Loss")
    ax.legend()
    path = FIG_DIR / "01_loss_curves.png"
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved → {path}")


def plot_cer(df):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df["epoch"], df["train_cer"], label="Train CER", color=STYLE_COLOR)
    ax.plot(df["epoch"], df["val_cer"],   label="Val CER",   color=STYLE_ORANGE)
    best_epoch = df.loc[df["val_cer"].idxmin(), "epoch"]
    best_cer   = df["val_cer"].min()
    ax.axvline(best_epoch, color="grey", linestyle="--", linewidth=0.8,
               label=f"Best val epoch {best_epoch} (CER={best_cer:.4f})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Character Error Rate")
    ax.set_title("Training vs Validation CER")
    ax.legend()
    path = FIG_DIR / "02_cer_curves.png"
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved → {path}")


def plot_lr(df):
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.semilogy(df["epoch"], df["lr"].astype(float), color=STYLE_COLOR)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Learning Rate (log scale)")
    ax.set_title("Learning-Rate Schedule (ReduceLROnPlateau)")
    path = FIG_DIR / "03_lr_schedule.png"
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved → {path}")


def plot_cer_histogram(results_df):
    """Overlay greedy and beam CER distributions as step histograms."""
    g_cer = results_df["greedy_cer"].astype(float).values
    b_cer = results_df["beam_cer"].astype(float).values

    bins = np.linspace(0, 1, 41)    # 40 bins of width 0.025
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(g_cer, bins=bins, alpha=0.6, label="Greedy",      color=STYLE_COLOR,  histtype="stepfilled")
    ax.hist(b_cer, bins=bins, alpha=0.6, label=f"Beam search", color=STYLE_ORANGE, histtype="stepfilled")
    ax.set_xlabel("Character Error Rate")
    ax.set_ylabel("Number of samples")
    ax.set_title("Test-Set CER Distribution")
    ax.legend()

    # Annotate median lines
    for arr, col, label in [(g_cer, STYLE_COLOR, "Greedy"), (b_cer, STYLE_ORANGE, "Beam")]:
        med = np.median(arr)
        ax.axvline(med, color=col, linestyle="--", linewidth=1.2)
        ax.text(med + 0.01, ax.get_ylim()[1] * 0.9, f"{label} median\n{med:.3f}",
                color=col, fontsize=8)

    path = FIG_DIR / "04_cer_histogram.png"
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved → {path}")


def plot_confusion(results_df):
    """
    Compute the top-20 most frequent character-level substitution errors
    (beam predictions vs ground truth) and display as a bar chart.

    We align each (prediction, ground_truth) pair using a simple
    sequence alignment and count substitution pairs.
    """
    errors: Counter = Counter()

    def align_chars(pred, gt):
        """
        Extract substitution pairs via DP alignment.
        Insertions / deletions are skipped.
        Returns list of (pred_char, gt_char) substitution pairs.
        """
        m, n = len(pred), len(gt)
        # DP table
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m + 1): dp[i][0] = i
        for j in range(n + 1): dp[0][j] = j
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                dp[i][j] = (dp[i-1][j-1] if pred[i-1] == gt[j-1]
                             else 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1]))
        # Traceback
        pairs = []
        i, j = m, n
        while i > 0 and j > 0:
            if pred[i-1] == gt[j-1]:
                i -= 1; j -= 1
            elif dp[i-1][j-1] <= dp[i-1][j] and dp[i-1][j-1] <= dp[i][j-1]:
                pairs.append((pred[i-1], gt[j-1]))   # substitution
                i -= 1; j -= 1
            elif dp[i-1][j] <= dp[i][j-1]:
                i -= 1                                 # deletion
            else:
                j -= 1                                 # insertion
        return pairs

    for _, row in results_df.iterrows():
        gt   = str(row["ground_truth"])
        pred = str(row["beam_pred"])
        for pc, gc in align_chars(pred, gt):
            if pc != gc:
                errors[(gc, pc)] += 1   # (correct → predicted_as)

    if not errors:
        print("  No substitution errors found — skipping confusion chart.")
        return

    top = errors.most_common(20)
    labels = [f"'{gt}'→'{pr}'" for (gt, pr), _ in top]
    counts = [c for _, c in top]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels[::-1], counts[::-1], color=STYLE_COLOR)
    ax.set_xlabel("Frequency")
    ax.set_title("Top-20 Character Substitution Errors (beam predictions)")
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    path = FIG_DIR / "05_error_analysis.png"
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved → {path}")


def main():
    print("── Visualisation ─────────────────────────────────────────────────")

    # Training curves (require training_log.csv)
    if LOG_CSV.exists():
        df = load_log()
        print(f"  training_log.csv: {len(df)} epochs")
        plot_loss(df)
        plot_cer(df)
        plot_lr(df)
    else:
        print(f"  [skip] {LOG_CSV} not found — run 2_train.py first")

    # Test analysis (require test_results.csv)
    if RESULTS_CSV.exists():
        res = pd.read_csv(RESULTS_CSV)
        res.columns = [c.strip() for c in res.columns]
        print(f"  test_results.csv: {len(res)} samples")
        plot_cer_histogram(res)
        plot_confusion(res)
    else:
        print(f"  [skip] {RESULTS_CSV} not found — run 3_test.py first")

    print(f"\nAll figures saved to  {FIG_DIR}/")


if __name__ == "__main__":
    main()